# OmegaConf / Hydra workshop (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willpenhall/ml-workshop-university-texeira/blob/main/stock_workshop/workshop_colab.ipynb)

This notebook contains the whole `stock_workshop` folder, so nothing else needs to be uploaded.

**Run the cells in section 0 first** (or use *Runtime → Run all*). They install the libraries and write the config files, `engine.py` and `run.py` into Colab's file browser (folder icon on the left). Anywhere the workshop says to run a command in the terminal, this version gives you a ready-made `!python run.py ...` cell instead.

*Generated by `build_colab.py` from the repo files. Edit those, not this notebook.*

## 0. Setup

In [ ]:
%pip install -q omegaconf hydra-core

In [ ]:
import os
from pathlib import Path

# Work inside ./stock_workshop (Colab starts in /content). Safe to re-run.
if Path.cwd().name != "stock_workshop":
    os.makedirs("stock_workshop", exist_ok=True)
    os.chdir("stock_workshop")
for group in ("market", "strategy", "backtest"):
    os.makedirs(f"conf/{group}", exist_ok=True)
print("Working directory:", Path.cwd())

### 0.1 Workspace files
`conf/config.yaml` is the Hydra default list. Each group folder (`market`, `strategy`, `backtest`) holds the options for that group. `engine.py` is the backtest engine and `run.py` is the Hydra entry point.

In [ ]:
%%writefile conf/config.yaml
defaults:
  - market: aapl
  - strategy: sma_crossover
  - backtest: default
  - _self_

In [ ]:
%%writefile conf/market/aapl.yaml
# conf/market/aapl.yaml
ticker: AAPL
start_price: 180.0
annual_drift: 0.08
annual_vol: 0.25
days: 252
seed: 42

In [ ]:
%%writefile conf/market/msft.yaml
# conf/market/msft.yaml  (same keys)
ticker: MSFT
start_price: 400.0
annual_drift: 0.10
annual_vol: 0.22
days: 252
seed: 42

In [ ]:
%%writefile conf/market/tsla.yaml
# conf/market/tsla.yaml  (same keys)
ticker: TSLA
start_price: 240.0
annual_drift: 0.12
annual_vol: 0.55
days: 252
seed: 42

In [ ]:
%%writefile conf/strategy/sma_crossover.yaml
# conf/strategy/sma_crossover.yaml
name: sma_crossover
fast_window: 10
slow_window: 30

In [ ]:
%%writefile conf/strategy/sma_ratio.yaml
# conf/strategy/sma_ratio.yaml
# slow_window is computed by the "mul" resolver registered in run.py
name: sma_ratio
fast_window: 10
ratio: 3
slow_window: ${mul:${.fast_window},${.ratio}}

In [ ]:
%%writefile conf/backtest/default.yaml
# conf/backtest/default.yaml
initial_cash: 10000.0
commission: 0.001
metrics:
  - total_return
  - max_drawdown

In [ ]:
%%writefile engine.py
"""Tiny backtest engine for the config workshop. Standard library only."""
import math
import random


def simulate_prices(start_price, annual_drift, annual_vol, days, seed):
    """Geometric random walk: one simulated closing price per trading day."""
    rng = random.Random(seed)
    dt = 1 / 252
    prices = [float(start_price)]
    for _ in range(days):
        shock = rng.gauss(0, 1)
        growth = (annual_drift - 0.5 * annual_vol ** 2) * dt + annual_vol * math.sqrt(dt) * shock
        prices.append(prices[-1] * math.exp(growth))
    return prices


def sma(prices, window, i):
    """Simple moving average of the last `window` prices ending at index i."""
    return sum(prices[i - window + 1 : i + 1]) / window


def run_backtest(prices, fast_window, slow_window, initial_cash, commission):
    """Moving-average crossover: all in when fast > slow, all cash when fast < slow."""
    cash = float(initial_cash)
    shares = 0.0
    equity_curve = []
    for i, price in enumerate(prices):
        if i >= slow_window - 1:
            fast = sma(prices, fast_window, i)
            slow = sma(prices, slow_window, i)
            if fast > slow and shares == 0:
                shares = cash * (1 - commission) / price
                cash = 0.0
            elif fast < slow and shares > 0:
                cash = shares * price * (1 - commission)
                shares = 0.0
        equity_curve.append(cash + shares * price)
    return equity_curve


def total_return(equity_curve):
    return equity_curve[-1] / equity_curve[0] - 1


def max_drawdown(equity_curve):
    peak = equity_curve[0]
    worst = 0.0
    for value in equity_curve:
        peak = max(peak, value)
        worst = min(worst, value / peak - 1)
    return worst


METRICS = {"total_return": total_return, "max_drawdown": max_drawdown}

In [ ]:
%%writefile run.py
import hydra
from omegaconf import DictConfig, OmegaConf

from engine import METRICS, run_backtest, simulate_prices

# Custom resolver: lets a config call Python, e.g. slow_window: ${mul:${.fast_window},3}
# Registered before @hydra.main so every YAML file in conf/ can use it.
OmegaConf.register_new_resolver("mul", lambda a, b: a * b)


@hydra.main(version_base=None, config_path="conf", config_name="config")
def main(cfg: DictConfig) -> None:
    print(OmegaConf.to_yaml(cfg))

    # A schema checks types, not relationships between values:
    assert cfg.strategy.fast_window < cfg.strategy.slow_window, \
        "fast_window must be smaller than slow_window"

    m, s, b = cfg.market, cfg.strategy, cfg.backtest
    prices = simulate_prices(m.start_price, m.annual_drift, m.annual_vol, m.days, m.seed)
    curve = run_backtest(prices, s.fast_window, s.slow_window, b.initial_cash, b.commission)

    results = ", ".join(f"{name}={METRICS[name](curve):+.1%}" for name in b.metrics)
    print(f"RESULT {m.ticker} sma {s.fast_window}/{s.slow_window}: {results} | final ${curve[-1]:,.0f}")


if __name__ == "__main__":
    main()

In [ ]:
# Check: every file is in place and the engine imports
for rel in ['conf/config.yaml', 'conf/market/aapl.yaml', 'conf/market/msft.yaml', 'conf/market/tsla.yaml', 'conf/strategy/sma_crossover.yaml', 'conf/strategy/sma_ratio.yaml', 'conf/backtest/default.yaml', 'engine.py', 'run.py']:
    assert Path(rel).is_file(), f"missing {rel}"
import engine
print("All files written. Simulated", len(engine.simulate_prices(100, 0.08, 0.25, 252, 42)), "prices.")

**1. Import different config files** 
<br> <br>
We will start the notebook with OmegaConfig, which exists to combine multiple configuration files into one tree. This optimizes workflow and reduces the need for a crowded hierarchy of config files. Features include adding to the tree with dotlist notation, merging with overrides, and checking variables against a typed schema.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from omegaconf import OmegaConf

base_dir = Path.cwd()

market = OmegaConf.load(str(base_dir / "conf" / "market" / "aapl.yaml"))              # stock data
strategy = OmegaConf.load(str(base_dir / "conf" / "strategy" / "sma_crossover.yaml"))  # details in engine.py
backtest = OmegaConf.load(str(base_dir / "conf" / "backtest" / "default.yaml"))        # initial stock account parameters

# tree made from market.yaml, strategy.yaml, backtest.yaml
cfg = OmegaConf.create({"market": market, "strategy": strategy, "backtest": backtest})
print(OmegaConf.to_yaml(cfg))
print(cfg.market.ticker, cfg.strategy.fast_window)

**2. Merging with override**
<br> <br>
Instead of manually editing the file architecture within VS code, you can add additional parameters to your config files straight from the notebook. It also saves the need for the "get" function when pulling the hyperparameter values into your model code.

In [ ]:
# formatting: config_name.param = value
dotlist = ["strategy.fast_window=5", "backtest.commission=0.002"]
override = OmegaConf.from_dotlist(dotlist)

# add/merge to existing config tree
cfg = OmegaConf.merge(cfg, override)

print(cfg.strategy.fast_window, cfg.backtest.commission)   # 5 0.002

**3. Checking variable types against a schema**
<br> <br>

This is an extra feature included with OmegaConf that ensures hyperparameter variable types, and throws errors early if a type mismatch exists.

In [ ]:
# Creates an unstructured list of variable types for each hyperparameter
@dataclass
class StrategyConfig:
    name: str
    fast_window: int
    slow_window: int

schema = OmegaConf.structured(StrategyConfig)

# Merge is unsuccessful if schema and strategy config variable types disagree
checked = OmegaConf.merge(schema, strategy)                # passes
print(checked.slow_window)                                 # 30

The next two cells fail on purpose. Each one prints the error OmegaConf raises instead of stopping the notebook.

In [ ]:
# ValidationError: "quick" is not an int
try:
    OmegaConf.merge(schema, {"fast_window": "quick"})
except Exception as err:
    print(f"{type(err).__name__}: {err}")

In [ ]:
# ConfigKeyError: typo caught
try:
    OmegaConf.merge(schema, {"fast_windw": 5})
except Exception as err:
    print(f"{type(err).__name__}: {err}")

**4. Hydra: groups and CLI overrides** \
\
Omegaconf has a built in configuration management tool called Hydra. it takes the config tree from a default list, gives it to the funtion, and lets the command long override anything. The folder names are the config groups (market, strategy, and backtest).\
\
If you look in the default.yaml, you will see the primary market, strategy, and backtest configs. Each line means "take conf/'group'/'option'.yaml and put it under the key 'group'". That is why segment 2 wrapped each file in market, strategy and backtest by hand.\
\
Now we head over to the run.py file, where using 'python run.py' in the terminal will output the results of the run using the primary configs.\
\
Output: RESULT AAPL sma 10/30: total_return=+26.6%, max_drawdown=-9.6% | final $12,658.

In [ ]:
!python run.py

**4.1.  Override from the CLI**\
\
Run each line and read the output before the next:

In [ ]:
# change values
!python run.py strategy.fast_window=5 backtest.commission=0.002

In [ ]:
# swap a whole group option
!python run.py market=tsla

In [ ]:
# + adds a key that does not exist
!python run.py +backtest.note=trial1

In [ ]:
# trips the fast < slow check
!python run.py strategy.fast_window=50

In [ ]:
# print the config, do not run main()
!python run.py --cfg job

**5: Multirun sweep**\
\
The -m flag runs run.py once per comma-separated value, with no change to the code.\
\
1: Launching 3 jobs locally, then three result lines identical to the ones in the Overview\
\
2: Sweep over the fast_window parameter with a value of 5, 10, and 20\
\
3: Sweep over two parameters, market and fast_window, with 2 different options each. This produces 4 different outcomes.

In [ ]:
!python run.py -m market=aapl,msft,tsla

In [ ]:
!python run.py -m strategy.fast_window=5,10,20

In [ ]:
!python run.py -m market=aapl,tsla strategy.fast_window=5,20

**6. Custom resolvers: running Python inside a config**
<br> <br>
OmegaConf lets a config value call a Python function. You register the function once with `OmegaConf.register_new_resolver("name", function)`, and then any value written as `${name:arg1,arg2}` calls it. Arguments can be plain numbers or strings, or other config values: `${.fast_window}` means "fast_window in this same section".
<br> <br>
The function runs every time the value is read, not when the file is loaded, so a derived value stays correct after an override. ML and RL training configs use this for hyperparameters that depend on other hyperparameters, such as a slow window tied to a fast window, a learning rate scaled by batch size, or an exploration rate that decays as training goes on.

In [ ]:
# Register Python functions that configs can call by name.
# replace=True lets you re-run this cell without an "already registered" error.
OmegaConf.register_new_resolver("mul", lambda a, b: a * b, replace=True)
OmegaConf.register_new_resolver("daily_vol", lambda annual_vol: round(annual_vol / 252 ** 0.5, 4), replace=True)

derived = OmegaConf.create("""
market:
  annual_vol: 0.25
  daily_vol: ${daily_vol:${.annual_vol}}     # annual volatility -> per-day volatility
strategy:
  fast_window: 10
  slow_window: ${mul:${.fast_window},3}      # always 3x the fast window
""")

print(derived.market.daily_vol)       # 0.0157
print(derived.strategy.slow_window)   # 30

# Resolvers run when the value is read, so derived values follow overrides:
derived.strategy.fast_window = 5
print(derived.strategy.slow_window)   # 15

**6.1. A training schedule (RL example)**
<br> <br>
An epsilon-greedy RL agent picks a random action with probability epsilon. Epsilon starts high so the agent explores, then decays as training goes on so it exploits what it has learned. A resolver lets the config hold the whole schedule: the training loop only updates the step counter and reads `epsilon`.

In [ ]:
def linear_decay(start, end, step, total_steps):
    """Move from start to end over total_steps, then hold at end."""
    progress = min(step / total_steps, 1.0)
    return round(start + (end - start) * progress, 3)

OmegaConf.register_new_resolver("linear_decay", linear_decay, replace=True)

rl = OmegaConf.create({
    "train": {"step": 0, "total_steps": 10_000},
    "explore": {"epsilon": "${linear_decay:1.0,0.05,${train.step},${train.total_steps}}"},
})

for step in [0, 2_500, 5_000, 10_000, 15_000]:
    rl.train.step = step                       # what a training loop does each step
    print(f"step {step:>6}: epsilon = {rl.explore.epsilon}")

**6.2. Resolvers with Hydra**
<br> <br>
`run.py` registers the same `mul` resolver before Hydra loads the configs, so any YAML file in `conf/` can use it. The new `conf/strategy/sma_ratio.yaml` sets the slow window as a multiple of the fast window:

```yaml
name: sma_ratio
fast_window: 10
ratio: 3
slow_window: ${mul:${.fast_window},${.ratio}}
```

Overriding `fast_window` now moves `slow_window` along with it, and a multirun sweep keeps the 1:3 ratio in every job. The config that `run.py` prints still shows the `${mul:...}` text, because it is printed before anything is resolved. Add `--resolve` after `--cfg job` to see the computed values.

In [ ]:
# same result as the default: 10/30
!python run.py strategy=sma_ratio

In [ ]:
# slow_window follows: 5/15
!python run.py strategy=sma_ratio strategy.fast_window=5

In [ ]:
# print computed values
!python run.py strategy=sma_ratio strategy.fast_window=5 --cfg job --resolve

In [ ]:
# sweep keeps the 1:3 ratio
!python run.py -m strategy=sma_ratio strategy.fast_window=5,10,20

In [ ]:
# slow == fast trips the fast < slow check
!python run.py strategy=sma_ratio strategy.ratio=1

**7: Q&A, troubleshooting**\
\
...